# Cross-user sEMG gesture recognition: does contrastive pretraining close the generalization gap?

**Author:** Jahid Hasan Jony · short self-directed exploration (Nov 2026)

EMG gesture decoders that work well for the people they were trained on often fail on new users, because muscle anatomy, electrode contact and armband placement differ. This notebook measures that gap on a public dataset and tests one idea from my thesis work on multivariate time series:

1. **Baseline:** a 1D-CNN trained with labels only.
2. **Contrastive pretraining (SimCLR-style)** with augmentations that mimic real EMG nuisance factors: per-channel gain (electrode contact), electrode rotation around the armband, channel dropout, noise and time shift. Then fine-tune with labels.
3. **Evaluation:** seen users on a new session vs. **unseen users** (6-fold, grouped by subject), with per-subject bootstrap confidence intervals.
4. **Explainability:** Integrated Gradients for channel/time attribution, with a completeness check and a deletion test for faithfulness.

**Data:** UCI *EMG Data for Gestures* (Myo armband, 8 channels, 36 subjects; Lobov et al., 2018). Runs locally in VS Code (uses the Apple Silicon GPU via MPS if available) or on Colab with a T4 GPU.

## 1. Setup

In [ ]:
import os, re, glob, zipfile, urllib.request, random, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

if torch.cuda.is_available():
    DEVICE = "cuda"                      # Colab GPU / NVIDIA
elif torch.backends.mps.is_available():
    DEVICE = "mps"                       # Apple Silicon Mac (M1/M2/M3/M4)
else:
    DEVICE = "cpu"
print("Device:", DEVICE, "(on Colab, switch to a T4 GPU if this says cpu)")

CFG = dict(
    WIN=200,            # window length in samples
    STEP=50,            # window hop
    N_FOLDS=6,          # cross-user folds (6 subjects held out per fold)
    SUP_EPOCHS=15,
    PRE_EPOCHS=20,      # contrastive pretraining epochs
    FT_EPOCHS=10,       # fine-tuning epochs after pretraining
    BATCH=256,
    LR=1e-3,
    TAU=0.1,            # NT-Xent temperature
    QUICK=False,        # True = fewer epochs/folds for a fast smoke test
)
if CFG["QUICK"]:
    CFG.update(SUP_EPOCHS=3, PRE_EPOCHS=3, FT_EPOCHS=2, N_FOLDS=3)
os.makedirs("results", exist_ok=True)
CFG

## 2. Download the dataset

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/481/emg+data+for+gestures.zip"
os.makedirs("data", exist_ok=True)
zp = "data/emg.zip"
if not os.path.exists(zp):
    urllib.request.urlretrieve(URL, zp)
with zipfile.ZipFile(zp) as z:
    z.extractall("data")
# some UCI archives contain a nested zip
for inner in glob.glob("data/**/*.zip", recursive=True):
    if os.path.abspath(inner) != os.path.abspath(zp):
        with zipfile.ZipFile(inner) as z:
            z.extractall(os.path.dirname(inner))

FILES = sorted(f for f in glob.glob("data/**/*.txt", recursive=True)
               if "readme" not in os.path.basename(f).lower())
print(len(FILES), "recording files, e.g.", FILES[:2])

## 3. Load recordings and cut windows

Each subject has two recording series (sessions). Labels: 0 = unmarked, 1 = rest, 2 = fist, 3 = wrist flexion, 4 = wrist extension, 5 = radial deviation, 6 = ulnar deviation, 7 = extended palm (only some subjects). I keep classes 1–6 and cut windows that lie entirely inside one gesture segment.

In [ ]:
def subject_of(path):
    m = re.search(r"(\d+)[\\/][^\\/]+\.txt$", path)
    return int(m.group(1)) if m else -1

def load_file(path):
    df = pd.read_csv(path, sep=r"\s+")
    df.columns = [str(c).lower() for c in df.columns]
    ch = [c for c in df.columns if c.startswith("channel")]
    return df[ch].to_numpy(np.float32), df["class"].to_numpy(int)

def segments(labels):
    # contiguous runs of the same label: (start, end, label)
    change = np.flatnonzero(np.diff(labels)) + 1
    starts = np.r_[0, change]; ends = np.r_[change, len(labels)]
    return [(s, e, labels[s]) for s, e in zip(starts, ends)]

def build_windows(files, win, step, keep=(1, 2, 3, 4, 5, 6)):
    X, y, subj, sess = [], [], [], []
    by_subject = {}
    for f in files:
        by_subject.setdefault(subject_of(f), []).append(f)
    for s, fs in sorted(by_subject.items()):
        for k, f in enumerate(sorted(fs)):
            sig, lab = load_file(f)
            for a, b, c in segments(lab):
                if c not in keep or b - a < win:
                    continue
                for t in range(a, b - win + 1, step):
                    X.append(sig[t:t + win].T); y.append(keep.index(c))
                    subj.append(s); sess.append(k)
    return (np.stack(X).astype(np.float32), np.array(y), np.array(subj), np.array(sess))

X, y, SUBJ, SESS = build_windows(FILES, CFG["WIN"], CFG["STEP"])
N_CLASSES = int(y.max()) + 1
GESTURES = ["rest", "fist", "wrist flexion", "wrist extension", "radial dev.", "ulnar dev."][:N_CLASSES]
print("windows:", X.shape, "| subjects:", len(np.unique(SUBJ)), "| classes:", N_CLASSES)
print("windows per class:", np.bincount(y))

## 4. Evaluation helpers (bootstrap CIs over subjects)

In [ ]:
def bootstrap_ci(values, n_boot=5000, alpha=0.05, seed=SEED):
    # percentile CI of the mean, resampling subjects
    v = np.asarray(values, float); rng = np.random.default_rng(seed)
    means = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(1)
    return v.mean(), np.quantile(means, alpha / 2), np.quantile(means, 1 - alpha / 2)

def per_subject_scores(y_true, y_pred, subj):
    rows = []
    for s in np.unique(subj):
        m = subj == s
        rows.append(dict(subject=int(s), acc=accuracy_score(y_true[m], y_pred[m]),
                         f1=f1_score(y_true[m], y_pred[m], average="macro")))
    return pd.DataFrame(rows)

def fmt(values):
    m, lo, hi = bootstrap_ci(values)
    return f"{100*m:.1f}% [95% CI {100*lo:.1f}–{100*hi:.1f}]"

## 5. Model, augmentations and training loops

In [ ]:
class Encoder(nn.Module):
    def __init__(self, c_in=8, d=128):
        super().__init__()
        def block(i, o, k):
            return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2), nn.BatchNorm1d(o), nn.GELU())
        self.net = nn.Sequential(block(c_in, 64, 7), nn.MaxPool1d(2),
                                 block(64, 128, 5), nn.MaxPool1d(2),
                                 block(128, d, 3), nn.AdaptiveAvgPool1d(1), nn.Flatten())
    def forward(self, x):
        return self.net(x)

class Classifier(nn.Module):
    def __init__(self, n_classes, c_in=8, d=128):
        super().__init__()
        self.enc = Encoder(c_in, d); self.head = nn.Linear(d, n_classes)
    def forward(self, x):
        return self.head(self.enc(x))

def augment(x):
    # EMG-specific nuisance factors. x: (B, C, T)
    B, C, T = x.shape
    x = x * (1 + 0.25 * torch.randn(B, C, 1, device=x.device))            # electrode contact / gain
    shift = torch.randint(-1, 2, (B,), device=x.device)                    # armband rotation by ±1 electrode
    idx = (torch.arange(C, device=x.device)[None] - shift[:, None]) % C
    x = torch.gather(x, 1, idx[:, :, None].expand(B, C, T))
    x = x * (torch.rand(B, C, 1, device=x.device) > 0.1).float()          # channel dropout
    x = torch.roll(x, shifts=int(torch.randint(-T // 8, T // 8 + 1, (1,))), dims=2)  # time shift
    return x + 0.05 * torch.randn_like(x)                                  # sensor noise

def nt_xent(z1, z2, tau):
    z = F.normalize(torch.cat([z1, z2]), dim=1)
    sim = z @ z.T / tau
    n = z1.shape[0]
    sim = sim.masked_fill(torch.eye(2 * n, dtype=torch.bool, device=z.device), float("-inf"))
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)

def batches(n, bs, shuffle=True):
    idx = np.random.permutation(n) if shuffle else np.arange(n)
    for i in range(0, n, bs):
        yield idx[i:i + bs]

def train_supervised(model, Xtr, ytr, epochs, lr, aug=False):
    model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    Xt, yt = torch.tensor(Xtr), torch.tensor(ytr)
    for ep in range(epochs):
        for b in batches(len(Xt), CFG["BATCH"]):
            if len(b) < 2: continue  # BatchNorm needs >1 sample
            xb, yb = Xt[b].to(DEVICE), yt[b].to(DEVICE)
            if aug: xb = augment(xb)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def pretrain_contrastive(enc, Xun, epochs, lr, tau):
    proj = nn.Sequential(nn.Linear(128, 128), nn.GELU(), nn.Linear(128, 64)).to(DEVICE)
    enc.to(DEVICE).train()
    opt = torch.optim.AdamW(list(enc.parameters()) + list(proj.parameters()), lr=lr, weight_decay=1e-4)
    Xt = torch.tensor(Xun)
    for ep in range(epochs):
        tot = 0.0
        for b in batches(len(Xt), CFG["BATCH"]):
            if len(b) < 2: continue
            xb = Xt[b].to(DEVICE)
            loss = nt_xent(proj(enc(augment(xb))), proj(enc(augment(xb))), tau)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
        if ep == 0 or (ep + 1) % 5 == 0:
            print(f"   pretrain epoch {ep+1}/{epochs}  NT-Xent {tot/len(Xt):.3f}")
    return enc

@torch.no_grad()
def predict(model, Xte):
    model.to(DEVICE).eval(); out = []
    Xt = torch.tensor(Xte)
    for b in batches(len(Xt), 1024, shuffle=False):
        out.append(model(Xt[b].to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(out)

def standardize(Xtr, *others):
    mu = Xtr.mean(axis=(0, 2), keepdims=True); sd = Xtr.std(axis=(0, 2), keepdims=True) + 1e-8
    return [(a - mu) / sd for a in (Xtr, *others)]

## 6. Experiment A: seen users, new session

Train on session 1 of every subject, test on session 2 of the same subjects. This is the *optimistic* setting most demos report.

In [ ]:
t0 = time.time(); set_seed()
tr, te = SESS == 0, SESS == 1
Xa_tr, Xa_te = standardize(X[tr], X[te])
m = train_supervised(Classifier(N_CLASSES), Xa_tr, y[tr], CFG["SUP_EPOCHS"], CFG["LR"])
seen = per_subject_scores(y[te], predict(m, Xa_te), SUBJ[te])
print("Seen users, new session  accuracy:", fmt(seen.acc), "| macro-F1:", fmt(seen.f1))
print(f"({time.time()-t0:.0f}s)")

## 7. Experiment B: unseen users (grouped cross-validation)

Whole subjects are held out. For each fold I compare:
- **baseline**: supervised only
- **baseline+aug**: supervised with the same augmentations (so the contrastive gain is not just augmentation)
- **contrastive**: SimCLR pretraining on training users, then fine-tuning
- **contrastive+target**: pretraining also sees the *unlabeled* windows of the held-out users (unsupervised adaptation, no labels from new users)

In [ ]:
METHODS = ["baseline", "baseline+aug", "contrastive", "contrastive+target"]
preds = {k: np.empty_like(y) for k in METHODS}
fold_models = {}
gkf = GroupKFold(n_splits=CFG["N_FOLDS"])
t0 = time.time()
for fold, (tri, tei) in enumerate(gkf.split(X, y, SUBJ)):
    print(f"Fold {fold+1}/{CFG['N_FOLDS']}: held-out subjects {sorted(set(SUBJ[tei].tolist()))}")
    Xtr, Xte = standardize(X[tri], X[tei])
    set_seed(SEED + fold)
    preds["baseline"][tei] = predict(train_supervised(Classifier(N_CLASSES), Xtr, y[tri], CFG["SUP_EPOCHS"], CFG["LR"]), Xte)
    set_seed(SEED + fold)
    preds["baseline+aug"][tei] = predict(train_supervised(Classifier(N_CLASSES), Xtr, y[tri], CFG["SUP_EPOCHS"], CFG["LR"], aug=True), Xte)
    for name, Xun in [("contrastive", Xtr), ("contrastive+target", np.concatenate([Xtr, Xte]))]:
        set_seed(SEED + fold)
        clf = Classifier(N_CLASSES)
        pretrain_contrastive(clf.enc, Xun, CFG["PRE_EPOCHS"], CFG["LR"], CFG["TAU"])
        train_supervised(clf, Xtr, y[tri], CFG["FT_EPOCHS"], CFG["LR"] / 3, aug=True)
        preds[name][tei] = predict(clf, Xte)
        if fold == 0 and name == "contrastive":
            fold_models["ig"] = (clf, Xte, y[tei])
    print(f"   done ({time.time()-t0:.0f}s elapsed)")

scores = {k: per_subject_scores(y, preds[k], SUBJ) for k in METHODS}
print("\nUnseen users (mean over subjects, bootstrap 95% CI):")
for k in METHODS:
    print(f"  {k:<20s} accuracy {fmt(scores[k].acc)} | macro-F1 {fmt(scores[k].f1)}")

## 8. Is the difference real? Paired bootstrap over subjects

In [ ]:
def paired(a, b):
    d = scores[a].acc.values - scores[b].acc.values
    m, lo, hi = bootstrap_ci(d)
    better = (d > 0).sum()
    return f"{a} − {b}: {100*m:+.1f} pts [95% CI {100*lo:+.1f}, {100*hi:+.1f}], better for {better}/{len(d)} subjects"

print(f"Generalization gap (seen-user session 2 vs unseen users, baseline): {100*(seen.acc.mean()-scores['baseline'].acc.mean()):.1f} pts")
for a, b in [("baseline+aug", "baseline"), ("contrastive", "baseline+aug"),
             ("contrastive+target", "contrastive"), ("contrastive+target", "baseline")]:
    print(paired(a, b))

## 9. Per-subject results

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
order = scores["baseline"].sort_values("acc").subject.values
xs = np.arange(len(order))
for i, k in enumerate(METHODS):
    s = scores[k].set_index("subject").loc[order, "acc"].values
    ax.bar(xs + (i - 1.5) * 0.2, 100 * s, width=0.2, label=k)
ax.axhline(100 * seen.acc.mean(), ls="--", c="k", lw=1, label="seen users (session 2)")
ax.set_xticks(xs); ax.set_xticklabels(order, fontsize=7)
ax.set_xlabel("held-out subject"); ax.set_ylabel("accuracy (%)"); ax.legend(fontsize=8, ncol=5)
ax.set_title("Accuracy on unseen users, per subject")
plt.tight_layout(); plt.savefig("results/per_subject_accuracy.png", dpi=150); plt.show()

## 10. Explainability: Integrated Gradients with faithfulness checks

Which electrodes and time points drive each gesture prediction on unseen users? I check that the attributions are faithful in two ways:
- **Completeness:** attributions should sum to f(x) − f(baseline) for the predicted logit.
- **Deletion test:** removing the most-attributed inputs should hurt accuracy faster than removing random inputs.

In [ ]:
def integrated_gradients(model, x, target, steps=32):
    model.eval(); base = torch.zeros_like(x); total = torch.zeros_like(x)
    for a in torch.linspace(0, 1, steps, device=x.device):
        xi = (base + a * (x - base)).requires_grad_(True)
        sel = model(xi).gather(1, target[:, None]).sum()
        total += torch.autograd.grad(sel, xi)[0]
    attr = (x - base) * total / steps
    with torch.no_grad():
        gap = (model(x) - model(base)).gather(1, target[:, None]).squeeze(1)
    return attr, gap

clf, Xte, yte = fold_models["ig"]
clf.to(DEVICE).eval()
rng = np.random.default_rng(SEED)
sel = rng.choice(len(Xte), size=min(600, len(Xte)), replace=False)
xb = torch.tensor(Xte[sel]).to(DEVICE)
with torch.no_grad():
    pred = clf(xb).argmax(1)
attrs, gaps = [], []
for b in batches(len(xb), 100, shuffle=False):
    a, g = integrated_gradients(clf, xb[b], pred[b]); attrs.append(a.detach()); gaps.append(g)
A = torch.cat(attrs); G = torch.cat(gaps)
rel_err = ((A.sum(dim=(1, 2)) - G).abs() / (G.abs() + 1e-8)).median().item()
print(f"Completeness: median relative error {100*rel_err:.1f}% (smaller is better; raise steps if large)")

# deletion test
fracs = [0, 0.05, 0.1, 0.2, 0.3, 0.5]
flatA = A.abs().flatten(1)
ytrue = torch.tensor(yte[sel]).to(DEVICE)
del_ig, del_rand = [], []
with torch.no_grad():
    for f in fracs:
        k = int(f * flatA.shape[1])
        for store, scoremat in [(del_ig, flatA), (del_rand, torch.rand_like(flatA))]:
            xm = xb.clone().flatten(1)
            if k > 0:
                top = scoremat.topk(k, dim=1).indices
                xm.scatter_(1, top, 0.0)
            store.append((clf(xm.view_as(xb)).argmax(1) == ytrue).float().mean().item())
plt.figure(figsize=(5, 3.5))
plt.plot([100*f for f in fracs], [100*v for v in del_ig], "o-", label="remove top-IG inputs")
plt.plot([100*f for f in fracs], [100*v for v in del_rand], "s--", label="remove random inputs")
plt.xlabel("% of input values removed"); plt.ylabel("accuracy (%)"); plt.legend(); plt.title("Deletion test")
plt.tight_layout(); plt.savefig("results/deletion_test.png", dpi=150); plt.show()

# channel importance per gesture
imp = A.abs().sum(2).cpu().numpy()          # (N, C)
pnp = pred.cpu().numpy()
H = np.stack([imp[pnp == c].mean(0) if (pnp == c).any() else np.zeros(imp.shape[1]) for c in range(N_CLASSES)])
H = H / (H.sum(1, keepdims=True) + 1e-12)
plt.figure(figsize=(6.5, 3.5))
plt.imshow(H, aspect="auto", cmap="viridis")
plt.colorbar(label="share of |attribution|")
plt.yticks(range(N_CLASSES), GESTURES)
plt.xticks(range(H.shape[1]), [f"ch{c+1}" for c in range(H.shape[1])])
plt.title("Electrode importance per predicted gesture (unseen users)")
plt.tight_layout(); plt.savefig("results/channel_importance.png", dpi=150); plt.show()

## 11. Save results and summary

In [ ]:
summary = dict(
    config=CFG,
    n_windows=int(len(X)), n_subjects=int(len(np.unique(SUBJ))),
    seen_users_session2=dict(zip(["mean", "lo", "hi"], map(float, bootstrap_ci(seen.acc)))),
    unseen_users={k: dict(zip(["mean", "lo", "hi"], map(float, bootstrap_ci(scores[k].acc)))) for k in METHODS},
    ig_completeness_median_rel_err=rel_err,
    deletion=dict(fracs=fracs, ig=del_ig, random=del_rand),
)
json.dump(summary, open("results/summary.json", "w"), indent=2)
pd.concat({k: v.set_index("subject") for k, v in scores.items()}, axis=1).to_csv("results/per_subject_scores.csv")

s, u = summary["seen_users_session2"], summary["unseen_users"]
print("Draft sentence for the motivation letter (check before using):\n")
print(f"On a public 8-channel sEMG dataset (36 users), gesture accuracy fell from {100*s['mean']:.0f}% on "
      f"seen users to {100*u['baseline']['mean']:.0f}% on unseen users; EMG-specific contrastive pretraining "
      f"gave {100*u['contrastive']['mean']:.0f}%, and {100*u['contrastive+target']['mean']:.0f}% when it also used "
      f"unlabeled data from the new users.")

import shutil; shutil.make_archive("emg_results", "zip", "results")
print("\nDownload emg_results.zip from the Files panel (left sidebar).")

## 12. Notes and limitations
- Short exploration on one dataset with a small CNN; results are indicative, not a benchmark.
- Hyperparameters were not tuned per method; `baseline+aug` is included so the contrastive effect is not confused with augmentation.
- `contrastive+target` uses unlabeled windows from held-out users (transductive adaptation); it never sees their labels.
- Next steps I would try: per-user calibration with a handful of labeled windows, domain-adversarial training, and combining EMG with egocentric video as weak supervision for intent.